In [1]:
# 必要パッケージの読み込み
if (!require("mice")) install.packages("mice")
if (!require("ggplot2")) install.packages("ggplot2")
if (!require("stringr")) install.packages("stringr")
if (!require("parallel")) install.packages("parallel") # 並列処理用

library(mice)
library(ggplot2)
library(stringr)
library(parallel)

# --- 1. 設定 -----------------------------------------------------------------
base_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/20251216_for_making_collect_data_nm_baseall_OHFP_csv/Cal _regression/MICE_check"
setwd(base_path)

files <- c(
  "Data20211217Final.csvrfN4maxit5R0.9893.csv",
  "Data20211217Final.csvcartN10maxit4R0.9791new.csv",
  "Data20211217Final.csvpmmN8maxit7R0.8281.csv"
)

n_trials <- 10
n_mask_cells <- 50

# 使用可能なCPUコア数の確認（全コア使うと重くなるため、-1程度が推奨）
n_cores <- max(1, parallel::detectCores() - 1)
cat(sprintf("Using %d cores for parallel processing.\n", n_cores))

# --- 2. パラメータ抽出関数 ---------------------------------------------------
get_params_from_name <- function(fname) {
  method <- ifelse(str_detect(fname, "rf"), "rf",
            ifelse(str_detect(fname, "cart"), "cart",
            ifelse(str_detect(fname, "pmm"), "pmm", "")))
  m_val <- as.numeric(str_extract(fname, "(?<=N)\\d+"))
  maxit_val <- as.numeric(str_extract(fname, "(?<=maxit)\\d+"))
  return(list(method = method, m = m_val, maxit = maxit_val))
}

# --- 3. 1回分の試行を実行する関数（並列化の単位） ---------------------------
run_single_trial <- function(trial_idx, df_orig, params, num_col_names) {
  # 各並列プロセス内でmiceを読み込む必要がある場合があります
  library(mice)
  
  # 数値が入っている場所を特定して50個マスク
  num_cols_data <- as.matrix(df_orig[, num_col_names])
  obs_idx <- which(!is.na(num_cols_data), arr.ind = TRUE)
  
  set.seed(trial_idx * 1000)
  mask_sel <- obs_idx[sample(nrow(obs_idx), 50), ]
  
  df_masked <- df_orig
  ground_truth <- numeric(50)
  
  for(i in 1:50) {
    r <- mask_sel[i, 1]; c_idx <- mask_sel[i, 2]
    c_name <- num_col_names[c_idx]
    ground_truth[i] <- df_orig[r, c_name]
    df_masked[r, c_name] <- NA
  }
  
  # MICE実行
  imp <- mice(df_masked, m = params$m, maxit = params$maxit, 
              method = params$method, printFlag = FALSE)
  df_complete <- complete(imp)
  
  # 誤差計算
  imputed_vals <- numeric(50)
  for(i in 1:50) {
    r <- mask_sel[i, 1]; c_name <- num_col_names[mask_sel[i, 2]]
    imputed_vals[i] <- df_complete[r, c_name]
  }
  
  rmse <- sqrt(mean((ground_truth - imputed_vals)^2, na.rm = TRUE))
  mae  <- mean(abs(ground_truth - imputed_vals), na.rm = TRUE)
  cor_val <- cor(ground_truth, imputed_vals, use = "complete.obs")
  
  return(data.frame(Trial = trial_idx, RMSE = rmse, MAE = mae, Correlation = cor_val))
}

# --- 4. メイン処理ループ -----------------------------------------------------
all_results <- data.frame()

for (f in files) {
  if (!file.exists(f)) next
  
  params <- get_params_from_name(f)
  cat(sprintf("\nProcessing %s in parallel...\n", f))
  
  df_orig <- read.csv(f, row.names = 1, stringsAsFactors = TRUE)
  num_cols <- sapply(df_orig, is.numeric)
  num_col_names <- names(df_orig)[num_cols]

  # 並列実行 (mclapply は Linux/Mac 用。Windowsの場合は parLapply を使用)
  # 10回の試行を各コアに割り振る
  trial_results_list <- mclapply(
    1:n_trials, 
    run_single_trial, 
    df_orig = df_orig, 
    params = params, 
    num_col_names = num_col_names, 
    mc.cores = n_cores,
    mc.set.seed = TRUE
  )
  
  # リストをデータフレームに変換
  trial_results_df <- do.call(rbind, trial_results_list)
  trial_results_df$Method <- params$method
  trial_results_df$FileName <- f
  
  all_results <- rbind(all_results, trial_results_df)
}

# --- 5. 保存と可視化 ---------------------------------------------------------
# 平均値などのサマリー作成
summary_stats <- aggregate(cbind(RMSE, MAE, Correlation) ~ Method, data = all_results, FUN = mean)
print(summary_stats)

write.csv(all_results, "MICE_Parallel_Full_Results.csv", row.names = FALSE)

# ボックスプロット
p <- ggplot(all_results, aes(x = Method, y = RMSE, fill = Method)) +
  geom_boxplot(alpha = 0.7) +
  geom_jitter(width = 0.1) +
  theme_minimal() +
  labs(title = "MICE Accuracy Comparison (Parallel Processing)",
       subtitle = "10 Trials per method, 50 random cells masked",
       y = "RMSE (Lower is better)")

print(p)
ggsave("MICE_Parallel_Accuracy_Boxplot.png", plot = p, width = 10, height = 6)

Loading required package: mice


Attaching package: 'mice'


The following object is masked from 'package:stats':

    filter


The following objects are masked from 'package:base':

    cbind, rbind


Loading required package: ggplot2

Loading required package: stringr

Loading required package: parallel



Using 63 cores for parallel processing.
